In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE databricks_301775.gold.dim_stations AS
    SELECT
        CAST(station_id AS STRING) AS station_id,
        TRIM(name) AS name,
        CAST(REPLACE(latitude, '_', '.') AS DOUBLE) AS latitude,
        CAST(REPLACE(longitude, '_', '.') AS DOUBLE) AS longitude
    FROM databricks_301775.silver.stations
    WHERE station_id IS NOT NULL
""")
print("✓ dim_stations created")

✓ dim_stations created


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE databricks_301775.gold.dim_riders AS
    SELECT
        CAST(rider_id AS INT) AS rider_id,
        TRIM(first) AS first_name,
        TRIM(last) AS last_name,
        TRIM(address) AS address,
        CAST(birthday AS DATE) AS birthday,
        CAST(account_start_date AS DATE) AS account_start_date,
        CAST(account_end_date AS DATE) AS account_end_date,
        CAST(is_member AS BOOLEAN) AS is_member
    FROM databricks_301775.silver.riders
    WHERE rider_id IS NOT NULL
""")
print("✓ dim_riders created")

✓ dim_riders created


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE databricks_301775.gold.dim_date AS
    WITH all_dates AS (
        SELECT DISTINCT CAST(started_at AS DATE) AS full_date
        FROM databricks_301775.silver.trips
        UNION
        SELECT DISTINCT CAST(date AS DATE) AS full_date
        FROM databricks_301775.silver.payments
    )
    SELECT
        CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
        full_date,
        dayofweek(full_date) AS day_of_week,
        date_format(full_date, 'EEEE') AS day_name,
        month(full_date) AS month,
        date_format(full_date, 'MMMM') AS month_name,
        quarter(full_date) AS quarter,
        year(full_date) AS year,
        CASE WHEN dayofweek(full_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
    FROM all_dates
    WHERE full_date IS NOT NULL
    ORDER BY full_date
""")
print("✓ dim_date created")

✓ dim_date created


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE databricks_301775.gold.fact_trips AS
    SELECT
        t.trip_id,
        CAST(t.rider_id AS STRING) AS rider_id,
        CAST(t.start_station_id AS STRING) AS start_station_id,
        CAST(t.end_station_id AS STRING) AS end_station_id,
        CAST(t.started_at AS TIMESTAMP) AS started_at,
        CAST(t.ended_at AS TIMESTAMP) AS ended_at,
        TRIM(t.rideable_type) AS rideable_type,
        (unix_timestamp(t.ended_at) - unix_timestamp(t.started_at)) / 60.0 AS ride_duration_minutes,
        CAST(date_format(CAST(t.started_at AS DATE), 'yyyyMMdd') AS INT) AS date_key,
        dayofweek(CAST(t.started_at AS TIMESTAMP)) AS day_of_week,
        hour(CAST(t.started_at AS TIMESTAMP)) AS hour_of_day,
        FLOOR(DATEDIFF(CAST(t.started_at AS DATE), CAST(r.birthday AS DATE)) / 365.25) AS rider_age_at_ride,
        CAST(r.is_member AS BOOLEAN) AS is_member
    FROM databricks_301775.silver.trips t
    JOIN databricks_301775.silver.riders r ON t.rider_id = r.rider_id
    WHERE t.trip_id IS NOT NULL
""")
print("✓ fact_trips created")

✓ fact_trips created


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE databricks_301775.gold.fact_payments AS
    SELECT
        CAST(p.payment_id AS INT) AS payment_id,
        CAST(p.rider_id AS INT) AS rider_id,
        CAST(p.date AS DATE) AS payment_date,
        CAST(REPLACE(p.amount, '_', '.') AS DOUBLE) AS amount,
        CAST(date_format(CAST(p.date AS DATE), 'yyyyMMdd') AS INT) AS date_key,
        month(CAST(p.date AS DATE)) AS month,
        quarter(CAST(p.date AS DATE)) AS quarter,
        year(CAST(p.date AS DATE)) AS year,
        FLOOR(DATEDIFF(CAST(p.date AS DATE), CAST(r.account_start_date AS DATE)) / 365.25) AS account_tenure_years,
        CAST(r.is_member AS BOOLEAN) AS is_member
    FROM databricks_301775.silver.payments p
    JOIN databricks_301775.silver.riders r ON p.rider_id = r.rider_id
    WHERE p.payment_id IS NOT NULL
""")
print("✓ fact_payments created")

✓ fact_payments created
